In [2]:
import operator
from typing import Annotated, List, TypedDict, Literal
from pydantic import BaseModel, Field

# The State object that travels through the graph
class AgentState(TypedDict):
    task: str
    research_notes: Annotated[List[str], operator.add]
    draft: str
    next_node: str
    retry_count: int
    revision_feedback: str

# Schema for the Supervisor to follow
class Router(BaseModel):
    """Decide which worker to call next."""
    next_worker: Literal["researcher", "writer", "FINISH"] = Field(description="The next node to act")
    instructions: str = Field(description="Specific instructions for the worker")
    is_critical: bool = Field(description="If True, system will pause for human review")

In [8]:
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults

# Using Llama 3.3 70B for the "Brain" (Supervisor)
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0)
search_tool = TavilySearchResults(k=2)

def researcher(state: AgentState):
    print("🚀 Researcher is digging...")
    query = state['task']
    results = search_tool.invoke(query)
    print(results)
    return {"research_notes": [str(results)], "retry_count": 0}

def writer(state: AgentState):
    print("✍️ Writer is composing...")
    context = "\n".join(state['research_notes'])
    res = llm.invoke(f"Write a report on {state['task']} using: {context}")
    return {"draft": res.content}

def supervisor(state: AgentState):
    print("🧠 Supervisor is reviewing state...")
    # Groq supports structured output natively
    structured_llm = llm.with_structured_output(Router)

    prompt = f"""
    Task: {state['task']}
    Notes collected: {len(state['research_notes'])}
    Current Draft: {state['draft'][:100]}...
    If you have something in research_notes, select writer
    """
    print(f"Supervisor sees {len(state['research_notes'])} notes and draft length {len(state['draft'])}")
    decision = structured_llm.invoke(prompt)
    return {
        "next_node": decision.next_worker,
        "revision_feedback": decision.instructions
    }

In [4]:
# Graph with recovery and breakpoints
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

builder = StateGraph(AgentState)

# Add nodes
builder.add_node("supervisor", supervisor)
builder.add_node("researcher", researcher)
builder.add_node("writer", writer)

# Logic
builder.set_entry_point("supervisor")

builder.add_conditional_edges(
    "supervisor",
    lambda x: x["next_node"],
    {"researcher": "researcher", "writer": "writer", "FINISH": END}
)

builder.add_edge("researcher", "supervisor")
builder.add_edge("writer", "supervisor")

# Enable Persistence (Recovery) and Breakpoints (Pause)
memory = MemorySaver()
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["writer"] # PAUSE: Allows human to review research before writing starts
)

In [9]:
config = {"configurable": {"thread_id": "workshop_user_1"}}
initial_input = {"task": "Impact of LPU architecture on AI inference speeds", "research_notes": [], "retry_count": 0, "draft": ""}

# 1. Start execution
print("--- STARTING GRAPH ---")
for event in graph.stream(initial_input, config, stream_mode="values"):
    if "next_node" in event:
        print(f"Moving to: {event['next_node']}")

# 2. Check if we hit a breakpoint (Pause)
snapshot = graph.get_state(config)
if snapshot.next:
    print(f"\n⏸️ SYSTEM PAUSED. Next step is: {snapshot.next}")
    print(f"Feedback from Supervisor: {snapshot.values['revision_feedback']}")

    # 3. Resume (Human says 'Go ahead')
    print("\n--- RESUMING AFTER PAUSE ---")
    #for event in graph.stream(None, config, stream_mode="values"):
    #    print("Finalizing...")

--- STARTING GRAPH ---
🧠 Supervisor is reviewing state...
Supervisor sees 0 notes and draft length 0
Moving to: researcher
🚀 Researcher is digging...
[{'title': 'LPU Unleashed: How Groq is Rewriting AI Inference', 'url': 'https://www.baytechconsulting.com/blog/lpu-unleashed-groq-rewriting-ai-inference', 'content': "This 10x architectural advantage allows the LPU to feed its compute units at maximum speed, entirely bypassing the memory bottleneck. The data movement happens instantaneously on the silicon, driving unprecedented Time-to-First-Token (TTFT) metrics and overall throughput speeds that frequently exceed human reading capabilities by an order of magnitude. From an environmental and operational perspective, this localized data movement also renders the LPU up to 10x more energy-efficient than traditional GPUs for inference workloads. Furthermore, the LPU and its accompanying GroqRack systems are air-cooled by design, eliminating the need for the complex, hyper-expensive liquid co

In [10]:
print("--- RESUMING GRAPH ---\n")
for event in graph.stream(None, config, stream_mode="values"):
    if "next_node" in event:
        print(f"Moving to: {event['next_node']}")
    elif "draft" in event:
        print(f"\n--- FINAL DRAFT ---:\n{event['draft']}")

--- RESUMING GRAPH ---

Moving to: writer
✍️ Writer is composing...
Moving to: writer
🧠 Supervisor is reviewing state...
Supervisor sees 1 notes and draft length 3414
Moving to: writer
